# DestinE ClimateDT Data Example

The Climate Change Adaptation Digital Twin (Climate DT) is the first ever attempt to produce multi-decadal climate projections operationally.

The Climate DT provides multi-decadal global climate simulations with local granularity, including information specific to the sectors most affected by climate change such as renewable energy, urban planning or hydrology. The objective is to produce updated simulations every year or less, compared to the current models, ran only every several years. This enables the inclusion of the latest developments in Earth system science and digital infrastructure.

The aim of this notebook is to provide some instructions on how to request access to the ClimateDT dataset and some examples on how to retrieve data.

## Requesting access to the data

Data from ClimateDT is available through the [Destination Earth Service Platform (DESP)](https://platform.destine.eu/).

Three steps are required in order to be able to follow this notebook and download data from the DESP

### Creating a DESP accoun

Users need to register at the DESP. The following [link](https://auth.destine.eu/realms/desp/protocol/openid-connect/auth) can be used.

### Asking for upgraded access

Users from academia and research need to ask for a upgraded access in order to use the Polytope service to access the data from Destination Earth. This can be requested in the following [link](https://platform.destine.eu/access-policy-upgrade/)

This process needs manual validation and may take some time.

### Generating the Polytope token.

Once upgraded access is granted the user needs to generate the Polytpe token that is used to authnticate the polytope requests. The token is generated by executing the following script: [desp_authentication.py](https://github.com/destination-earth-digital-twins/polytope-examples/blob/main/desp-authentication.py). A copy of the script has been added to the repository for convenience.

The script will ask for your DESP username and password and will generate the token file at `${HOME}/.polytopeapirc`.

The user may need ton install some Python dependencies in order to run the previous script. These dependencies are needed only for the token generation and no longer needed through the tutorial.

For adding the dependencies to the uv generated venv, open a new terminal and connect to Jusuf:

```
ssh -i ~/.ssh/id_ed25519_jsc USERNAME@jusuf.fz-juelich.de
cd compression-lab-notebooks
source .venv/bin/activate
uv pip install lxml
uv pip install conflator
```

## Downloading DesinE data with Polytope

The recommended way of getting the data is through `polytope`, interfaced with `earthkit-data`. This packages must be installed since they were not included in the `uv` environment provided in the original repo.

### Installing the missing packages

From another terminal, connect to the Jusuf machine (if working locally, this step can be skipped)

```
ssh -i ~/.ssh/id_ed25519_jsc USERNAME@jusuf.fz-juelich.de
```

*Note:* You might need to change the `~/.ssh/id_ed25519_jsc` value if you used another name for the SSH key added to your account.


Activate the virtual environment
```
cd compression-lab-notebooks
source .venv/bin/activate
```

Install the missing dependencies
```
uv pip install earthkit
```


## Downloading with earthkit

Data request to polytope are done with a MARS-like request syntax.

For more information about the **MARS keys needed** check [Data Structure and Keys](https://platform.destine.eu/services/documents-and-api/doc/?service_name=climate-dt-user-guide&doc_page=/models/IFS-NEMO/index.html)

For more information about the **available data variables** check [Data Catalogue](https://platform.destine.eu/services/documents-and-api/doc/?service_name=climate-dt-user-guide&doc_page=/models/IFS-NEMO/index.html)

In the following example, 1 day of an hourly field (`2t`, 2 metre temperature) is downloaded.

In [5]:
import earthkit.data

request = {
    'class': 'd1',  # Always d1 for DestinE data
    'dataset': 'climate-dt',  # Always climate-dt for ClimateDT data
    'experiment': 'ssp3-7.0',  # Depends of the experiment. Common options: cont, hist, ssp3-7.0, tplus2.0k...
    'activity': 'projections',  # projections for dates after 20150101, baseline for dates before 20150101
    'model': 'ifs-nemo',  # Available models: ifs-nemo, ifs-fesom, icon
    'generation': '2',   # Always 2 for DestinE generation 2 data.
    'realization': '1',  # Selects between members on an ensemble
    'stream': 'clte',  # clte for high frequency data (hourly for atmos, daily for ocean). clmn for monthly means
    'resolution': 'high',  # high for highest available resolution (H1024 or H512 depending on the experiment), standard for data interpolated to H128
    'type': 'fc',  # Always fc for ClimateDT data
    'expver': '0001',  # Always 0001 for production data
    'levtype': 'sfc',  # sfc: surface, pl: pressure levels, hl: height levels, sol: soil levels, o2d: ocean 2D, o3d: ocean 3D
    'param': '2t',  # Variable by GRIB short name or GRIB paramID (both are allowed)
    'date': '20200101',  # Specify date to request in YYYYMMDD format
    'time': '0000/to/2300/by/0100'  # Specify time to request in hhmm format
}

data_1hourly = earthkit.data.from_source(
   "polytope",  # Underlying access engine
   "destination-earth",  # Select dataset
   request,
   stream=False,
   address="polytope.mn5.apps.dte.destination-earth.eu"  # Point to Marenostrum5 DataBridge
)

2026-09-16 09:58:52 - INFO - Key read from /home/igonzal1/.polytopeapirc
2026-09-16 09:58:52 - INFO - Sending request...
{'request': 'activity: projections\n'
            'class: d1\n'
            'dataset: climate-dt\n'
            "date: '20200101'\n"
            'experiment: ssp3-7.0\n'
            "expver: '0001'\n"
            "generation: '2'\n"
            'levtype: sfc\n'
            'model: ifs-nemo\n'
            'param: 2t\n'
            "realization: '1'\n"
            'resolution: high\n'
            'stream: clte\n'
            'time: 0000/to/2300/by/0100\n'
            'type: fc\n',
 'verb': 'retrieve'}
2026-09-16 09:58:52 - INFO - Polytope user key found in session cache for user igonzal1
2026-09-16 09:58:53 - INFO - Request accepted. Please poll ./01f3z8k9836q3bt008hsg2n6ca for status
2026-09-16 09:58:53 - INFO - Polytope user key found in session cache for user igonzal1
2026-09-16 09:58:53 - INFO - Checking request status (01f3z8k9836q3bt008hsg2n6ca)...
2026-09-16 09:

01f3z8k9836q3bt008hsg2n6ca.grib:   0%|          | 0.00/600M [00:00<?, ?B/s]

Data is returned in an earthkit handle, that can be then decoded to xaray

In [4]:
data_1hourly.to_xarray()

<xarray.Dataset> Size: 3GB
Dimensions:                  (forecast_reference_time: 24, values: 12582912)
Coordinates:
  * forecast_reference_time  (forecast_reference_time) datetime64[ns] 192B 20...
    latitude                 (values) float64 101MB ...
    longitude                (values) float64 101MB ...
Dimensions without coordinates: values
Data variables:
    2t                       (forecast_reference_time, values) float64 2GB ...
Attributes:
    Conventions:  CF-1.8
    institution:  ECMWF

For 3D variables (like pressure levels), an additional, `levelist` key is needed in the request

In [6]:
import earthkit.data

request = {
    'class': 'd1',  # Always d1 for DestinE data
    'dataset': 'climate-dt',  # Always climate-dt for ClimateDT data
    'experiment': 'ssp3-7.0',  # Depends of the experiment. Common options: cont, hist, ssp3-7.0, tplus2.0k...
    'activity': 'projections',  # projections for dates after 20150101, baseline for dates before 20150101
    'model': 'ifs-nemo',  # Available models: ifs-nemo, ifs-fesom, icon
    'generation': '2',   # Always 2 for DestinE generation 2 data.
    'realization': '1',  # Selects between members on an ensemble
    'stream': 'clte',  # clte for high frequency data (hourly for atmos, daily for ocean). clmn for monthly means
    'resolution': 'high',  # high for highest available resolution (H1024 or H512 depending on the experiment), standard for data interpolated to H128
    'type': 'fc',  # Always fc for ClimateDT data
    'expver': '0001',  # Always 0001 for production data
    'levtype': 'pl',  # sfc: surface, pl: pressure levels, hl: height levels, sol: soil levels, o2d: ocean 2D, o3d: ocean 3D
    'param': 'u',  # Variable by GRIB short name or GRIB paramID (both are allowed)
    'date': '20200101',  # Specify date to request in YYYYMMDD format
    'time': '0000/to/2300/by/0600',  # Specify time to request in hhmm format
    'levelist': ['1000', '850']  # Pressure levels in hPa. Data available in plev19 scheme
}

data_pl_6hourly = earthkit.data.from_source(
   "polytope",
   "destination-earth",
   request,
   stream=False,
   address="polytope.mn5.apps.dte.destination-earth.eu")

2026-09-16 10:08:56 - INFO - Key read from /home/igonzal1/.polytopeapirc
2026-09-16 10:08:56 - INFO - Sending request...
{'request': 'activity: projections\n'
            'class: d1\n'
            'dataset: climate-dt\n'
            "date: '20200101'\n"
            'experiment: ssp3-7.0\n'
            "expver: '0001'\n"
            "generation: '2'\n"
            'levelist:\n'
            "- '1000'\n"
            "- '850'\n"
            'levtype: pl\n'
            'model: ifs-nemo\n'
            'param: u\n'
            "realization: '1'\n"
            'resolution: high\n'
            'stream: clte\n'
            'time: 0000/to/2300/by/0600\n'
            'type: fc\n',
 'verb': 'retrieve'}
2026-09-16 10:08:56 - INFO - Polytope user key found in session cache for user igonzal1
2026-09-16 10:08:56 - INFO - Request accepted. Please poll ./01f3z8k9836q4hg008z9rt04xh for status
2026-09-16 10:08:56 - INFO - Polytope user key found in session cache for user igonzal1
2026-09-16 10:08:56 - INFO

01f3z8k9836q4hg008z9rt04xh.grib:   0%|          | 0.00/217M [00:00<?, ?B/s]

In [7]:
data_pl_6hourly.to_xarray()

<xarray.Dataset> Size: 1GB
Dimensions:                  (forecast_reference_time: 4, level: 2,
                              values: 12582912)
Coordinates:
  * forecast_reference_time  (forecast_reference_time) datetime64[ns] 32B 202...
  * level                    (level) int64 16B 850 1000
    latitude                 (values) float64 101MB ...
    longitude                (values) float64 101MB ...
Dimensions without coordinates: values
Data variables:
    u                        (forecast_reference_time, level, values) float64 805MB ...
Attributes:
    Conventions:  CF-1.8
    institution:  ECMWF